In [ ]:
# Step 1 — Load targeted close-reading candidate pool
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "outputs"

candidate_path = OUTPUT_DIR / "step25_targeted_close_reading_pool.xlsx"

close_reading_pool = pd.read_excel(candidate_path)

print("Candidate annotations:", len(close_reading_pool))

print("\nColumns:")
for col in close_reading_pool.columns:
    print("-", col)

display(close_reading_pool.head())

In [ ]:
# Step 2 Inspect the targeted candidate pool
# Create a readable pattern label

close_reading_pool["pattern"] = (
    close_reading_pool["category"].astype(str)
    + " | "
    + close_reading_pool["justice_dimension"].astype(str)
    + " | "
    + close_reading_pool["institutional_role"].astype(str)
)

# Candidate distribution by institution
category_summary = (
    close_reading_pool
    .groupby("category")
    .agg(
        candidate_annotations=("annotation_id", "count"),
        unique_transcripts=("transcript_id", "nunique")
    )
    .reset_index()
)

print("Candidate distribution by institution:")
display(category_summary)

# Candidate distribution by priority pattern
pattern_summary = (
    close_reading_pool
    .groupby(
        ["category", "justice_dimension", "institutional_role"],
        as_index=False
    )
    .agg(
        candidate_annotations=("annotation_id", "count"),
        unique_transcripts=("transcript_id", "nunique")
    )
    .sort_values(
        ["category", "candidate_annotations"],
        ascending=[True, False]
    )
)

print("\nCandidate distribution by priority pattern:")
display(pattern_summary)

print("\nNumber of patterns:", len(pattern_summary))
print("Unique transcripts:", close_reading_pool["transcript_id"].nunique())

In [ ]:
# Step 3 Locate and verify source transcripts

from pathlib import Path
import re
import pandas as pd

TRANSCRIPT_DIR = (
    PROJECT_DIR /
    "dataset" /
    "Justice_GBV_VS_Interviews"
)

# Recursively locate both .doc and .docx files
transcript_files = [
    p for p in TRANSCRIPT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".doc", ".docx"}
]

print("Transcript root exists:", TRANSCRIPT_DIR.exists())
print("Total transcript files found:", len(transcript_files))

# Count formats
suffix_counts = pd.Series(
    [p.suffix.lower() for p in transcript_files]
).value_counts()

print("\nFile formats:")
display(suffix_counts.to_frame("count"))

# Extract JUSTxxx ID from filename
file_records = []

for p in transcript_files:
    match = re.search(r"(JUST\d+)", p.name, flags=re.IGNORECASE)

    file_records.append({
        "transcript_id": match.group(1).upper() if match else None,
        "filename": p.name,
        "suffix": p.suffix.lower(),
        "filepath": str(p)
    })

transcript_index = pd.DataFrame(file_records)

print("\nFirst 15 indexed files:")
display(transcript_index.head(15))

In [ ]:
# Match candidate transcript IDs to source files

candidate_ids = set(close_reading_pool["transcript_id"].astype(str).str.upper())
source_ids = set(transcript_index["transcript_id"].dropna())

matched_ids = candidate_ids & source_ids
missing_ids = candidate_ids - source_ids

print("Candidate transcript IDs:", len(candidate_ids))
print("Matched source transcripts:", len(matched_ids))
print("Missing source transcripts:", len(missing_ids))

if missing_ids:
    print("\nMissing IDs:")
    print(sorted(missing_ids))

In [ ]:
# Step 4 — Inspect existing processed corpus files

processed_files = [
    p for p in OUTPUT_DIR.rglob("*")
    if p.is_file()
    and any(term in p.name.lower() for term in [
        "corpus",
        "transcript",
        "clean",
        "master"
    ])
]

print("Potential processed corpus files:", len(processed_files))

for p in processed_files:
    print(p.name)

In [ ]:
# Step 5 — Inspect candidate transcript-text files

files_to_check = [
    OUTPUT_DIR / "transcripts_extracted_text.csv",
    OUTPUT_DIR / "transcript_texts_v2.csv"
]

for path in files_to_check:
    print("\n" + "=" * 70)
    print("FILE:", path.name)

    df_temp = pd.read_csv(path)

    print("Shape:", df_temp.shape)
    print("Columns:", df_temp.columns.tolist())

    display(df_temp.head(3))

In [ ]:
# Step 6 — Match candidate transcripts to processed transcript text

transcript_texts = pd.read_csv(
    OUTPUT_DIR / "transcript_texts_v2.csv"
)

# Create a simple transcript_id field
transcript_texts["transcript_id"] = (
    transcript_texts["participant_ids_str"]
    .astype(str)
    .str.upper()
    .str.strip()
)

candidate_ids = set(
    close_reading_pool["transcript_id"]
    .astype(str)
    .str.upper()
    .str.strip()
)

processed_ids = set(transcript_texts["transcript_id"])

matched_ids = candidate_ids & processed_ids
missing_ids = candidate_ids - processed_ids

print("Candidate transcript IDs:", len(candidate_ids))
print("Matched processed transcripts:", len(matched_ids))
print("Missing processed transcripts:", len(missing_ids))

if missing_ids:
    print("\nMissing IDs:")
    print(sorted(missing_ids))

# Check text availability for candidate transcripts
candidate_transcripts = transcript_texts[
    transcript_texts["transcript_id"].isin(candidate_ids)
].copy()

print("\nCandidate transcript rows:", len(candidate_transcripts))
print(
    "Missing text_clean:",
    candidate_transcripts["text_clean"].isna().sum()
)
print(
    "Missing text_raw:",
    candidate_transcripts["text_raw"].isna().sum()
)

display(
    candidate_transcripts[
        [
            "transcript_id",
            "file_name",
            "read_status",
            "partial_transcript",
            "interview_notes_no_audio",
            "non_standard_text_type"
        ]
    ].head(10)
)

In [ ]:
# Step 7 — Test whether annotation contexts can be relocated in transcript text

import re

# Create transcript_id -> text_clean lookup
text_lookup = (
    candidate_transcripts
    .set_index("transcript_id")["text_clean"]
    .to_dict()
)

def normalise_text(text):
    """Normalise whitespace for matching."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


match_results = []

for _, row in close_reading_pool.iterrows():

    transcript_id = str(row["transcript_id"]).upper().strip()
    context = normalise_text(row["context"])
    full_text = normalise_text(text_lookup.get(transcript_id, ""))

    exact_match = context in full_text if context else False

    match_results.append({
        "annotation_id": row["annotation_id"],
        "transcript_id": transcript_id,
        "exact_context_match": exact_match
    })


context_match_check = pd.DataFrame(match_results)

print("Annotation records:", len(context_match_check))
print(
    "Exact context matches:",
    context_match_check["exact_context_match"].sum()
)
print(
    "Unmatched:",
    (~context_match_check["exact_context_match"]).sum()
)

print(
    "\nExact match rate:",
    f"{context_match_check['exact_context_match'].mean():.1%}"
)

display(
    context_match_check[
        ~context_match_check["exact_context_match"]
    ].head(10)
)

In [ ]:
# Step 8 — Test keyword occurrence in source transcripts

keyword_results = []

for _, row in close_reading_pool.iterrows():

    transcript_id = str(row["transcript_id"]).upper().strip()
    keyword = normalise_text(row["keyword"])
    full_text = normalise_text(text_lookup.get(transcript_id, ""))

    # Count literal keyword occurrences
    keyword_count = full_text.count(keyword) if keyword else 0

    keyword_results.append({
        "annotation_id": row["annotation_id"],
        "transcript_id": transcript_id,
        "category": row["category"],
        "keyword": row["keyword"],
        "keyword_count": keyword_count
    })

keyword_check = pd.DataFrame(keyword_results)

print("Annotation records:", len(keyword_check))
print(
    "Keyword found in transcript:",
    (keyword_check["keyword_count"] > 0).sum()
)
print(
    "Keyword not found:",
    (keyword_check["keyword_count"] == 0).sum()
)

print("\nOccurrence-count distribution:")
display(
    keyword_check["keyword_count"]
    .value_counts()
    .sort_index()
    .rename_axis("keyword_count")
    .reset_index(name="annotations")
)

print("\nRecords where keyword was not found:")
display(
    keyword_check[
        keyword_check["keyword_count"] == 0
    ]
)

In [ ]:
# Step 9 — Inspect existing passage-level outputs

passage_files = [
    OUTPUT_DIR / "step9_passages_for_nmf.csv",
    OUTPUT_DIR / "step9_tfidf_documents.csv"
]

for path in passage_files:
    print("\n" + "=" * 80)
    print("FILE:", path.name)

    df_temp = pd.read_csv(path)

    print("Shape:", df_temp.shape)
    print("Columns:")
    for col in df_temp.columns:
        print("-", col)

    display(df_temp.head(5))

In [ ]:
# Step 10 — Generate existing passage candidates for each annotation

import ast

passages = pd.read_csv(
    OUTPUT_DIR / "step9_passages_for_nmf.csv"
)

# Standardise linking fields
passages["transcript_id"] = (
    passages["transcript_id"]
    .astype(str)
    .str.upper()
    .str.strip()
)

passages["category"] = (
    passages["category"]
    .astype(str)
    .str.strip()
)

def parse_keywords(value):
    """Convert stored keyword list to a normalised Python list."""
    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, list):
            return [normalise_text(x) for x in parsed]
    except Exception:
        pass

    return [normalise_text(value)]


passages["keyword_list"] = passages["keywords"].apply(parse_keywords)


link_results = []

for _, row in close_reading_pool.iterrows():

    annotation_id = row["annotation_id"]
    transcript_id = str(row["transcript_id"]).upper().strip()
    category = str(row["category"]).strip()
    keyword = normalise_text(row["keyword"])

    # First restrict to the same transcript and institution category
    candidates = passages[
        (passages["transcript_id"] == transcript_id) &
        (passages["category"] == category)
    ].copy()

    # Then check whether the annotation keyword occurs in
    # either the stored keyword list or the passage text
    if not candidates.empty:
        candidates["keyword_match"] = candidates.apply(
            lambda x:
                keyword in x["keyword_list"]
                or keyword in normalise_text(x["passage_text"]),
            axis=1
        )

        keyword_candidates = candidates[
            candidates["keyword_match"]
        ].copy()
    else:
        keyword_candidates = candidates.copy()

    link_results.append({
        "annotation_id": annotation_id,
        "transcript_id": transcript_id,
        "category": category,
        "keyword": row["keyword"],
        "same_transcript_category_passages": len(candidates),
        "keyword_matched_passages": len(keyword_candidates)
    })


passage_link_check = pd.DataFrame(link_results)

print("Annotation records:", len(passage_link_check))

print(
    "Annotations with >=1 keyword-matched passage:",
    (passage_link_check["keyword_matched_passages"] >= 1).sum()
)

print(
    "Annotations with exactly 1 keyword-matched passage:",
    (passage_link_check["keyword_matched_passages"] == 1).sum()
)

print(
    "Annotations with multiple keyword-matched passages:",
    (passage_link_check["keyword_matched_passages"] > 1).sum()
)

print(
    "Annotations with no keyword-matched passage:",
    (passage_link_check["keyword_matched_passages"] == 0).sum()
)

print("\nCandidate-count distribution:")

display(
    passage_link_check["keyword_matched_passages"]
    .value_counts()
    .sort_index()
    .rename_axis("matched_passages")
    .reset_index(name="annotations")
)

print("\nRecords requiring further disambiguation:")

display(
    passage_link_check[
        passage_link_check["keyword_matched_passages"] != 1
    ].sort_values(
        "keyword_matched_passages",
        ascending=False
    )
)

In [ ]:
# Step 11 — Rank candidate passages using annotation-context similarity

from difflib import SequenceMatcher
import pandas as pd

def context_similarity(annotation_context, passage_text):
    a = normalise_text(annotation_context)
    b = normalise_text(passage_text)

    if not a or not b:
        return 0.0

    # Direct containment is especially informative because annotation
    # contexts were originally derived from retrieved corpus text.
    if a in b:
        return 1.0

    if b in a:
        return len(b) / len(a)

    return SequenceMatcher(None, a, b).ratio()


ranked_candidates = []

for _, row in close_reading_pool.iterrows():

    annotation_id = row["annotation_id"]
    transcript_id = str(row["transcript_id"]).upper().strip()
    category = str(row["category"]).strip()
    keyword = normalise_text(row["keyword"])
    annotation_context = row["context"]

    candidates = passages[
        (passages["transcript_id"] == transcript_id) &
        (passages["category"] == category)
    ].copy()

    candidates["keyword_match"] = candidates.apply(
        lambda x:
            keyword in x["keyword_list"]
            or keyword in normalise_text(x["passage_text"]),
        axis=1
    )

    candidates = candidates[candidates["keyword_match"]].copy()

    candidates["similarity"] = candidates["passage_text"].apply(
        lambda x: context_similarity(annotation_context, x)
    )

    candidates = candidates.sort_values(
        "similarity",
        ascending=False
    ).reset_index(drop=True)

    for rank, (_, candidate) in enumerate(candidates.head(3).iterrows(), start=1):

        ranked_candidates.append({
            "annotation_id": annotation_id,
            "transcript_id": transcript_id,
            "category": category,
            "keyword": row["keyword"],
            "rank": rank,
            "similarity": candidate["similarity"],
            "passage_start": candidate["passage_start"],
            "passage_end": candidate["passage_end"],
            "n_windows": candidate["n_windows"],
            "passage_keywords": candidate["keywords"],
            "annotation_context": annotation_context,
            "passage_text": candidate["passage_text"]
        })


ranked_passages = pd.DataFrame(ranked_candidates)

# Produce one-row-per-annotation diagnostic summary
diagnostics = []

for annotation_id, group in ranked_passages.groupby("annotation_id"):

    group = group.sort_values("rank")

    top1 = group.iloc[0]["similarity"]

    if len(group) >= 2:
        top2 = group.iloc[1]["similarity"]
        margin = top1 - top2
    else:
        top2 = None
        margin = None

    diagnostics.append({
        "annotation_id": annotation_id,
        "top1_similarity": top1,
        "top2_similarity": top2,
        "top1_top2_margin": margin
    })

passage_match_diagnostics = pd.DataFrame(diagnostics)

print("Annotations ranked:", len(passage_match_diagnostics))

print("\nTop-1 similarity summary:")
display(
    passage_match_diagnostics["top1_similarity"]
    .describe()
    .to_frame()
)

print("\nLowest-confidence matches:")
display(
    passage_match_diagnostics
    .sort_values("top1_similarity")
    .head(15)
)

In [ ]:
# Step 12 — Inspect lower-confidence passage matches

REVIEW_THRESHOLD = 0.80

review_ids = (
    passage_match_diagnostics.loc[
        passage_match_diagnostics["top1_similarity"] < REVIEW_THRESHOLD,
        "annotation_id"
    ]
    .tolist()
)

print("Annotations requiring manual linkage review:", len(review_ids))
print(review_ids)

for annotation_id in review_ids:

    print("\n" + "=" * 110)
    print("ANNOTATION:", annotation_id)

    ann_row = close_reading_pool[
        close_reading_pool["annotation_id"] == annotation_id
    ].iloc[0]

    print("Transcript:", ann_row["transcript_id"])
    print("Category:", ann_row["category"])
    print("Keyword:", ann_row["keyword"])
    print("Institutional role:", ann_row["institutional_role"])
    print("Justice dimension:", ann_row["justice_dimension"])
    print("Evaluative direction:", ann_row["evaluative_direction"])

    print("\nANNOTATION CONTEXT:")
    print(ann_row["context"])

    candidates = (
        ranked_passages[
            ranked_passages["annotation_id"] == annotation_id
        ]
        .sort_values("rank")
        .head(3)
    )

    for _, candidate in candidates.iterrows():

        print("\n" + "-" * 90)
        print(
            f"RANK {int(candidate['rank'])} | "
            f"similarity = {candidate['similarity']:.3f} | "
            f"position = {candidate['passage_start']}–{candidate['passage_end']}"
        )

        print("PASSAGE:")
        print(candidate["passage_text"])

## Step 14 — Return to the original transcript context

The verified passage positions are used as anchors for returning to the fuller
transcript text. For each candidate, an expanded section of the original
transcript is extracted around the relevant passage.

The expanded window is used as a reading aid rather than as a fixed analytical
unit. Where the institutional encounter extends beyond the extracted window,
the surrounding transcript is consulted further so that interpretation follows
the narrative context rather than an arbitrary character boundary.

In [ ]:
# Step 14 — Extract expanded transcript context around verified passages

CONTEXT_EXTENSION = 1500

transcript_lookup = (
    candidate_transcripts
    .set_index("transcript_id")
)

close_reading_contexts = []

for _, row in verified_passage_links.iterrows():

    transcript_id = row["transcript_id"]

    if transcript_id not in transcript_lookup.index:
        print(f"WARNING: transcript not found: {transcript_id}")
        continue

    transcript_row = transcript_lookup.loc[transcript_id]

    text_clean = str(transcript_row["text_clean"])

    passage_start = int(row["passage_start"])
    passage_end = int(row["passage_end"])

    context_start = max(
        0,
        passage_start - CONTEXT_EXTENSION
    )

    context_end = min(
        len(text_clean),
        passage_end + CONTEXT_EXTENSION
    )

    expanded_context = text_clean[
        context_start:context_end
    ]

    close_reading_contexts.append({
        "annotation_id": row["annotation_id"],
        "transcript_id": transcript_id,
        "category": row["category"],
        "keyword": row["keyword"],
        "passage_start": passage_start,
        "passage_end": passage_end,
        "context_start": context_start,
        "context_end": context_end,
        "verified_passage": row["passage_text"],
        "expanded_context": expanded_context
    })

close_reading_contexts = pd.DataFrame(close_reading_contexts)

print("Close-reading contexts:", len(close_reading_contexts))
print(
    "Unique transcripts:",
    close_reading_contexts["transcript_id"].nunique()
)

print("\nExpanded context length:")
display(
    close_reading_contexts["expanded_context"]
    .str.len()
    .describe()
    .to_frame("characters")
)

## Step 15 — Construct the close-reading worksheet

The verified transcript contexts are combined with the existing manual
annotation fields to create a structured worksheet for targeted close reading.

The worksheet retains the earlier institutional-role, justice-dimension and
evaluative coding while adding fields for interpretive notes. These fields are
used to examine institutional action, survivors' descriptions of institutional
responses, narrative complexity, and the relationship between individual cases
and recurrent computational patterns.

Close reading remains interpretive rather than a further coding or
classification exercise.

In [ ]:
# Step 15 — Construct close-reading worksheet

annotation_fields = close_reading_pool[
    [
        "annotation_id",
        "institutional_role",
        "justice_dimension",
        "evaluative_direction",
        "dominant_topic",
        "dominant_topic_weight"
    ]
].copy()

close_reading_worksheet = close_reading_contexts.merge(
    annotation_fields,
    on="annotation_id",
    how="left",
    validate="one_to_one"
)

# Priority-pattern identifier
close_reading_worksheet["priority_pattern"] = (
    close_reading_worksheet["category"].astype(str)
    + " | "
    + close_reading_worksheet["justice_dimension"].astype(str)
    + " | "
    + close_reading_worksheet["institutional_role"].astype(str)
)

# Fields to be completed during close reading
close_reading_worksheet["institutional_action"] = ""
close_reading_worksheet["survivor_interpretation"] = ""
close_reading_worksheet["justice_interpretation"] = ""
close_reading_worksheet["ambiguity_or_sequence"] = ""
close_reading_worksheet["pattern_relationship"] = ""
close_reading_worksheet["analytical_value"] = ""
close_reading_worksheet["use_in_results"] = ""

print("Worksheet records:", len(close_reading_worksheet))
print(
    "Unique transcripts:",
    close_reading_worksheet["transcript_id"].nunique()
)
print(
    "Priority patterns:",
    close_reading_worksheet["priority_pattern"].nunique()
)

print("\nRecords by institution:")
display(
    close_reading_worksheet["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="records")
)

print("\nRecords by priority pattern:")
display(
    close_reading_worksheet[
        [
            "category",
            "justice_dimension",
            "institutional_role"
        ]
    ]
    .value_counts()
    .reset_index(name="records")
)

In [ ]:
# Step 15 — Construct and validate the close-reading worksheet

annotation_fields = close_reading_pool[
    [
        "annotation_id",
        "institutional_role",
        "justice_dimension",
        "evaluative_direction",
        "dominant_topic",
        "dominant_topic_weight",
        "mental_health_learning_need",
        "parental_status",
        "bme_status",
        "nationality_summary",
        "physical_disability"
    ]
].copy()

close_reading_worksheet = close_reading_contexts.merge(
    annotation_fields,
    on="annotation_id",
    how="left",
    validate="one_to_one"
)

# Priority-pattern identifier
close_reading_worksheet["priority_pattern"] = (
    close_reading_worksheet["category"].astype(str)
    + " | "
    + close_reading_worksheet["justice_dimension"].astype(str)
    + " | "
    + close_reading_worksheet["institutional_role"].astype(str)
)

# Fields to be completed during close reading
close_reading_worksheet["institutional_action"] = ""
close_reading_worksheet["survivor_interpretation"] = ""
close_reading_worksheet["justice_interpretation"] = ""
close_reading_worksheet["ambiguity_or_sequence"] = ""
close_reading_worksheet["pattern_relationship"] = ""
close_reading_worksheet["analytical_value"] = ""
close_reading_worksheet["use_in_results"] = ""

# Basic worksheet checks
print("Worksheet records:", len(close_reading_worksheet))
print(
    "Unique transcripts:",
    close_reading_worksheet["transcript_id"].nunique()
)
print(
    "Priority patterns:",
    close_reading_worksheet["priority_pattern"].nunique()
)

print("\nRecords by institution:")
display(
    close_reading_worksheet["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="records")
)

print("\nRecords by priority pattern:")
display(
    close_reading_worksheet[
        [
            "category",
            "justice_dimension",
            "institutional_role"
        ]
    ]
    .value_counts()
    .reset_index(name="records")
)

# Validate social-position fields
social_position_fields = [
    "mental_health_learning_need",
    "parental_status",
    "bme_status",
    "nationality_summary",
    "physical_disability"
]

print("\nSocial-position fields included:")
print(social_position_fields)

print("\nMissing values in social-position fields:")
display(
    close_reading_worksheet[
        social_position_fields
    ]
    .isna()
    .sum()
    .rename_axis("field")
    .reset_index(name="missing_values")
)

print(
    "\nDuplicate annotation IDs:",
    close_reading_worksheet["annotation_id"].duplicated().sum()
)

## Step 16 — Prepare the first-pass close-reading review

All 39 candidate annotations are retained for a first-pass review and arranged
within their 13 priority patterns. The review proceeds pattern by pattern so
that cases are compared with other candidates representing the same
institution–justice–role combination.

Each priority pattern will retain at least one primary case. Additional cases
may be retained where they provide contrast, qualification, narrative sequence,
institutional ambivalence, or relevant social-position context. The intended
final set is approximately 13–18 cases.

The first pass records concise observations rather than complete case
interpretations. Detailed justice interpretation and integration with the
computational findings are completed only for cases retained after this
comparison.

In [ ]:
# Step 16 — Prepare the first-pass close-reading review order

# Preserve the existing worksheet
first_pass_review = close_reading_worksheet.copy()

# Number of candidates available within each priority pattern
first_pass_review["pattern_candidate_n"] = (
    first_pass_review
    .groupby("priority_pattern")["annotation_id"]
    .transform("size")
)

# Administrative ordering for pattern-by-pattern review
category_order = {
    "legal": 1,
    "police": 2,
    "support_sector": 3
}

first_pass_review["category_order"] = (
    first_pass_review["category"]
    .map(category_order)
)

first_pass_review = (
    first_pass_review
    .sort_values(
        [
            "category_order",
            "justice_dimension",
            "institutional_role",
            "annotation_id"
        ]
    )
    .reset_index(drop=True)
)

# Sequential review number
first_pass_review["review_order"] = range(
    1,
    len(first_pass_review) + 1
)

# First-pass review fields
first_pass_review["review_complete"] = False
first_pass_review["first_pass_decision"] = ""
first_pass_review["first_pass_reason"] = ""

# Remove temporary ordering variable
first_pass_review = first_pass_review.drop(
    columns="category_order"
)

# Move administrative review fields near the front
front_columns = [
    "review_order",
    "annotation_id",
    "transcript_id",
    "category",
    "priority_pattern",
    "pattern_candidate_n",
    "institutional_role",
    "justice_dimension",
    "evaluative_direction"
]

remaining_columns = [
    col for col in first_pass_review.columns
    if col not in front_columns
]

first_pass_review = first_pass_review[
    front_columns + remaining_columns
]

print("First-pass records:", len(first_pass_review))
print(
    "Unique transcripts:",
    first_pass_review["transcript_id"].nunique()
)
print(
    "Priority patterns:",
    first_pass_review["priority_pattern"].nunique()
)

print(
    "Review-order range:",
    first_pass_review["review_order"].min(),
    "to",
    first_pass_review["review_order"].max()
)

print(
    "Duplicate review-order values:",
    first_pass_review["review_order"].duplicated().sum()
)

print("\nFirst-pass records by priority pattern:")
display(
    first_pass_review[
        [
            "category",
            "justice_dimension",
            "institutional_role",
            "pattern_candidate_n"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nFirst 10 records in review order:")
display(
    first_pass_review[
        [
            "review_order",
            "annotation_id",
            "transcript_id",
            "category",
            "justice_dimension",
            "institutional_role",
            "pattern_candidate_n"
        ]
    ]
    .head(10)
)

## Step 17 — Export the first-pass close-reading workbook

The ordered first-pass worksheet is exported to Excel for manual review.

The workbook preserves the traceability fields, existing annotation labels,
social-position context, verified source passages, and expanded transcript
context. It also includes the fields used to record concise first-pass
observations and provisional case-selection decisions.

The exported workbook is an intermediate analytical record. First-pass
decisions remain provisional until all candidates within the same priority
pattern have been compared.

In [ ]:
# Step 17 — Export the first-pass close-reading workbook

from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

first_pass_columns = [
    # Review identifiers
    "review_order",
    "annotation_id",
    "transcript_id",
    "category",
    "priority_pattern",
    "pattern_candidate_n",

    # Existing annotation fields
    "keyword",
    "institutional_role",
    "justice_dimension",
    "evaluative_direction",
    "dominant_topic",
    "dominant_topic_weight",

    # Social-position context
    "mental_health_learning_need",
    "parental_status",
    "bme_status",
    "nationality_summary",
    "physical_disability",

    # Source traceability
    "passage_start",
    "passage_end",
    "verified_passage",
    "expanded_context",

    # First-pass interpretive fields
    "institutional_action",
    "survivor_interpretation",
    "ambiguity_or_sequence",
    "analytical_value",
    "review_complete",
    "first_pass_decision",
    "first_pass_reason",

    # Reserved for selected cases
    "justice_interpretation",
    "pattern_relationship",
    "use_in_results"
]

# Create a separate export copy
first_pass_export = first_pass_review[
    first_pass_columns
].copy()


# Remove Excel-incompatible control characters
# from the export copy only
def clean_excel_text(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value


text_columns = first_pass_export.select_dtypes(
    include=["object"]
).columns

illegal_character_count = 0

for col in text_columns:

    illegal_character_count += (
        first_pass_export[col]
        .apply(
            lambda value: (
                len(ILLEGAL_CHARACTERS_RE.findall(value))
                if isinstance(value, str)
                else 0
            )
        )
        .sum()
    )

    first_pass_export[col] = (
        first_pass_export[col]
        .apply(clean_excel_text)
    )


# Coding guide for manual review
coding_guide = pd.DataFrame(
    [
        {
            "field": "institutional_action",
            "instruction": (
                "Briefly record what the institution or institutional actor "
                "did, failed to do, required, enabled, or prevented."
            )
        },
        {
            "field": "survivor_interpretation",
            "instruction": (
                "Record how the survivor describes or evaluates the "
                "institutional response. Do not infer an unstated view."
            )
        },
        {
            "field": "ambiguity_or_sequence",
            "instruction": (
                "Record mixed experiences, changes over time, contradictory "
                "responses, narrative sequence, or limits of the extracted context."
            )
        },
        {
            "field": "analytical_value",
            "instruction": (
                "State briefly what this case adds to understanding the "
                "priority pattern."
            )
        },
        {
            "field": "review_complete",
            "instruction": (
                "Enter TRUE only after the expanded context has been read."
            )
        },
        {
            "field": "first_pass_decision",
            "instruction": (
                "Use only: primary, contrast, reserve, or exclude."
            )
        },
        {
            "field": "first_pass_reason",
            "instruction": (
                "Give a concise evidence-based reason for the provisional "
                "selection decision."
            )
        },
        {
            "field": "social-position variables",
            "instruction": (
                "Use only as contextual or exploratory information where the "
                "narrative itself shows analytical relevance. Do not infer "
                "experience from metadata alone."
            )
        }
    ]
)

# Clean the coding guide as an additional safety check
for col in coding_guide.select_dtypes(include=["object"]).columns:
    coding_guide[col] = coding_guide[col].apply(
        clean_excel_text
    )


# Define output path
first_pass_output = (
    OUTPUT_DIR
    / "step17_first_pass_close_reading_workbook.xlsx"
)


# Export workbook
with pd.ExcelWriter(
    first_pass_output,
    engine="openpyxl"
) as writer:

    first_pass_export.to_excel(
        writer,
        sheet_name="first_pass_review",
        index=False
    )

    coding_guide.to_excel(
        writer,
        sheet_name="coding_guide",
        index=False
    )


# Validate exported workbook
print("First-pass workbook saved to:")
print(first_pass_output)

print(
    "\nExcel-incompatible characters removed:",
    illegal_character_count
)

print(
    "Exported records:",
    len(first_pass_export)
)

print(
    "Unique annotation IDs:",
    first_pass_export["annotation_id"].nunique()
)

print(
    "Priority patterns:",
    first_pass_export["priority_pattern"].nunique()
)

print(
    "Output file exists:",
    first_pass_output.exists()
)

print(
    "Output file size:",
    first_pass_output.stat().st_size
    if first_pass_output.exists()
    else None
)

In [ ]:
# Step 18 — Validate the exported close-reading workbook

workbook_file = pd.ExcelFile(first_pass_output)

print("Workbook sheets:")
print(workbook_file.sheet_names)

reloaded_first_pass = pd.read_excel(
    first_pass_output,
    sheet_name="first_pass_review"
)

reloaded_coding_guide = pd.read_excel(
    first_pass_output,
    sheet_name="coding_guide"
)

print("\nReloaded review records:", len(reloaded_first_pass))
print(
    "Unique annotation IDs:",
    reloaded_first_pass["annotation_id"].nunique()
)
print(
    "Unique transcripts:",
    reloaded_first_pass["transcript_id"].nunique()
)
print(
    "Priority patterns:",
    reloaded_first_pass["priority_pattern"].nunique()
)

print(
    "Duplicate annotation IDs:",
    reloaded_first_pass["annotation_id"].duplicated().sum()
)

print(
    "Missing expanded contexts:",
    reloaded_first_pass["expanded_context"].isna().sum()
)

print(
    "Missing verified passages:",
    reloaded_first_pass["verified_passage"].isna().sum()
)

print(
    "Annotation order preserved:",
    reloaded_first_pass["annotation_id"].tolist()
    == first_pass_export["annotation_id"].tolist()
)

print(
    "Annotation ID set preserved:",
    set(reloaded_first_pass["annotation_id"])
    == set(first_pass_export["annotation_id"])
)

print(
    "Expanded context preserved:",
    (
        reloaded_first_pass["expanded_context"]
        .fillna("")
        .astype(str)
        .tolist()
        ==
        first_pass_export["expanded_context"]
        .fillna("")
        .astype(str)
        .tolist()
    )
)

print(
    "Coding-guide rows:",
    len(reloaded_coding_guide)
)